# NDJF Gap-Merge Catalog Cleanup

This notebook builds merged candidate catalogs from the existing NDJF event table without rerunning the expensive ERA5 detection workflow.

In [4]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
# Keep the notebook source and helper modules on the same project branch.
BRANCH = "codex/notebook16-pcolormesh"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent output dir: /content/drive/MyDrive/JPCZcatalog_outputs
Removed existing repo clone: /content/JPCZcatalog

Cloning into '/content/JPCZcatalog'...

Working directory: /content/JPCZcatalog


In [5]:
from pathlib import Path
import shutil

import pandas as pd

from jpcz_catalog.detect import merge_gap_sensitivity, merge_nearby_events
from jpcz_catalog.verification import render_gap_merge_summary, write_text_summary

drive_catalog_path = Path(DRIVE_OUTPUT_DIR) / 'jpcz_catalog_ndjf.csv'
repo_catalog_path = Path('outputs/verification/jpcz_catalog_ndjf.csv')
CATALOG_PATH = drive_catalog_path if drive_catalog_path.exists() else repo_catalog_path
if not CATALOG_PATH.exists():
    raise FileNotFoundError(f'No raw NDJF catalog found at {drive_catalog_path} or {repo_catalog_path}. Run Notebook 04 first.')
print('Raw catalog input:', CATALOG_PATH)
MERGE_GAP_VALUES = (6, 12, 24)
RECOMMENDED_GAP_HOURS = 12

catalog_df = pd.read_csv(
    CATALOG_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)

sensitivity_df = merge_gap_sensitivity(
    catalog_df,
    gap_hours_values=MERGE_GAP_VALUES,
)
sensitivity_path = Path("outputs/verification/jpcz_catalog_ndjf_gap_merge_sensitivity.csv")
sensitivity_df.to_csv(sensitivity_path, index=False)

merged_catalog = merge_nearby_events(
    catalog_df,
    max_gap_hours=RECOMMENDED_GAP_HOURS,
)
merged_catalog_path = Path(f"outputs/verification/jpcz_catalog_ndjf_merged_{int(RECOMMENDED_GAP_HOURS)}h.csv")
merged_catalog.to_csv(merged_catalog_path, index=False)

summary_text = render_gap_merge_summary(
    sensitivity_df=sensitivity_df,
    recommended_gap_hours=RECOMMENDED_GAP_HOURS,
    merged_catalog_name=merged_catalog_path.name,
)
summary_path = write_text_summary(
    f"outputs/verification/jpcz_catalog_ndjf_merged_{int(RECOMMENDED_GAP_HOURS)}h_summary.md",
    summary_text,
)

if PERSIST_OUTPUTS_TO_DRIVE:
    for path in [sensitivity_path, merged_catalog_path, summary_path]:
        shutil.copy2(path, Path(DRIVE_OUTPUT_DIR) / path.name)

print(summary_text)
merged_catalog.head()


Raw catalog input: /content/drive/MyDrive/JPCZcatalog_outputs/jpcz_catalog_ndjf.csv
# NDJF Gap-Merge Sensitivity

The raw NDJF detector groups only strictly consecutive hourly threshold hits.
This sensitivity table shows how many events remain after merging adjacent events separated by small threshold-free gaps.

 gap_hours  event_count  events_merged
       0.0          287              0
       6.0          282              5
      12.0          275             12
      24.0          256             31

Recommended candidate: merge gaps <= 12 hours
- Resulting event count: 275
- Output catalog: `jpcz_catalog_ndjf_merged_12h.csv`

Interpretation:
A short-gap merge broadens the catalog from threshold-hit fragments toward synoptic episodes without rerunning the expensive ERA5 detection workflow.


,event_start,event_end,event_peak,event_peak_D_s-1,event_peak_D_1e5_s-1,duration_hours,year,month,season_year,detection_threshold_s-1,...,zeta_box_mean_s-1,slp_climatology_mean_hpa,slp_climatology_std_hpa,monsoon_type,shinoda_class,episode_span_hours,threshold_hit_hours,merged_subevents_count,total_internal_gap_hours,max_internal_gap_hours
0,2000-01-13 08:00:00,2000-01-13 14:00:00,2000-01-13 11:00:00,-0.000030,-2.972377,7,2000,1,2000,-0.000024,...,0.000007,10.914953,8.697740,Type 2 weak-monsoon,Type 2 weak-monsoon,7,7,1,0.0,0.0
1,2000-01-19 22:00:00,2000-01-20 08:00:00,2000-01-20 04:00:00,-0.000029,-2.927972,11,2000,1,2000,-0.000024,...,0.000016,10.914953,8.697740,Type 1 strong-monsoon,Type 1B higher-vorticity,11,11,1,0.0,0.0
2,2000-02-07 13:00:00,2000-02-08 13:00:00,2000-02-08 00:00:00,-0.000043,-4.312036,25,2000,2,2000,-0.000020,...,0.000014,8.781942,8.398704,Type 2 weak-monsoon,Type 2 weak-monsoon,25,25,1,0.0,0.0
3,2000-02-13 18:00:00,2000-02-16 01:00:00,2000-02-14 22:00:00,-0.000036,-3.603144,56,2000,2,2000,-0.000020,...,0.000004,8.781942,8.398704,Type 1 strong-monsoon,Type 1A lower-vorticity,56,47,2,10.0,10.0
4,2000-02-27 18:00:00,2000-02-28 04:00:00,2000-02-27 23:00:00,-0.000032,-3.168735,11,2000,2,2000,-0.000020,...,0.000015,8.781942,8.398704,Type 1 strong-monsoon,Type 1A lower-vorticity,11,11,1,0.0,0.0


## Read-through table: final merged events

Run the preceding merge cell first. This cell creates the full, numbered catalog used for review. It is saved to Google Drive as `jpcz_catalog_ndjf_merged_12h_readthrough.csv` and displayed below as a paged, searchable Colab table. Set one or more filters to inspect a particular year, month, or date range; leave all filters as `None` to browse every event.

In [6]:
# Optional filters for reviewing the final merged catalog.
YEAR_TO_VIEW = None          # e.g., 2023
MONTH_TO_VIEW = None         # e.g., 1 for January
START_DATE = None             # e.g., '2023-01-24' (UTC)
END_DATE = None               # e.g., '2023-01-26 23:59' (UTC)

catalog_view = merged_catalog.copy().sort_values('event_peak').reset_index(drop=True)
catalog_view.insert(0, 'event_number', catalog_view.index + 1)
for column in ['event_start', 'event_end', 'event_peak']:
    catalog_view[column] = pd.to_datetime(catalog_view[column])

# ERA5 timestamps are UTC. JST is included only to make visual event review easier.
catalog_view['event_start_jst'] = catalog_view['event_start'] + pd.Timedelta(hours=9)
catalog_view['event_end_jst'] = catalog_view['event_end'] + pd.Timedelta(hours=9)
catalog_view['event_peak_jst'] = catalog_view['event_peak'] + pd.Timedelta(hours=9)

readthrough_path = Path('outputs/verification/jpcz_catalog_ndjf_merged_12h_readthrough.csv')
catalog_view.to_csv(readthrough_path, index=False)
if PERSIST_OUTPUTS_TO_DRIVE:
    drive_readthrough_path = Path(DRIVE_OUTPUT_DIR) / readthrough_path.name
    shutil.copy2(readthrough_path, drive_readthrough_path)
    print('Saved full read-through CSV:', drive_readthrough_path)

review_view = catalog_view.copy()
if YEAR_TO_VIEW is not None:
    review_view = review_view[review_view['event_peak'].dt.year == int(YEAR_TO_VIEW)]
if MONTH_TO_VIEW is not None:
    review_view = review_view[review_view['event_peak'].dt.month == int(MONTH_TO_VIEW)]
if START_DATE is not None:
    review_view = review_view[review_view['event_peak'] >= pd.Timestamp(START_DATE)]
if END_DATE is not None:
    review_view = review_view[review_view['event_peak'] <= pd.Timestamp(END_DATE)]

display_columns = [
    'event_number', 'event_start', 'event_end', 'event_peak', 'event_peak_jst',
    'duration_hours', 'threshold_hit_hours', 'merged_subevents_count',
    'max_internal_gap_hours', 'event_peak_D_1e5_s-1',
    'monsoon_type', 'shinoda_class',
]
display_columns = [column for column in display_columns if column in review_view.columns]
print(f'Reviewing {len(review_view)} of {len(catalog_view)} final merged events.')
print('Peak-date coverage:', catalog_view['event_peak'].min(), 'to', catalog_view['event_peak'].max())

try:
    from google.colab import data_table
    display(data_table.DataTable(review_view[display_columns], include_index=False, num_rows_per_page=25))
except ImportError:
    display(review_view[display_columns])


Saved full read-through CSV: /content/drive/MyDrive/JPCZcatalog_outputs/jpcz_catalog_ndjf_merged_12h_readthrough.csv
Reviewing 275 of 275 final merged events.
Peak-date coverage: 2000-01-13 11:00:00 to 2025-12-25 14:00:00


,event_number,event_start,event_end,event_peak,event_peak_jst,duration_hours,threshold_hit_hours,merged_subevents_count,max_internal_gap_hours,event_peak_D_1e5_s-1,monsoon_type,shinoda_class
0,1,2000-01-13 08:00:00,2000-01-13 14:00:00,2000-01-13 11:00:00,2000-01-13 20:00:00,7,7,1,0.0,-2.972377,Type 2 weak-monsoon,Type 2 weak-monsoon
1,2,2000-01-19 22:00:00,2000-01-20 08:00:00,2000-01-20 04:00:00,2000-01-20 13:00:00,11,11,1,0.0,-2.927972,Type 1 strong-monsoon,Type 1B higher-vorticity
2,3,2000-02-07 13:00:00,2000-02-08 13:00:00,2000-02-08 00:00:00,2000-02-08 09:00:00,25,25,1,0.0,-4.312036,Type 2 weak-monsoon,Type 2 weak-monsoon
3,4,2000-02-13 18:00:00,2000-02-16 01:00:00,2000-02-14 22:00:00,2000-02-15 07:00:00,56,47,2,10.0,-3.603144,Type 1 strong-monsoon,Type 1A lower-vorticity
4,5,2000-02-27 18:00:00,2000-02-28 04:00:00,2000-02-27 23:00:00,2000-02-28 08:00:00,11,11,1,0.0,-3.168735,Type 1 strong-monsoon,Type 1A lower-vorticity
...,...,...,...,...,...,...,...,...,...,...,...,...
270,271,2025-02-12 15:00:00,2025-02-12 16:00:00,2025-02-12 15:00:00,2025-02-13 00:00:00,2,2,1,0.0,-2.040109,Type 1 strong-monsoon,Type 1A lower-vorticity
271,272,2025-11-27 14:00:00,2025-11-28 01:00:00,2025-11-27 17:00:00,2025-11-28 02:00:00,12,12,1,0.0,-2.486155,Type 2 weak-monsoon,Type 2 weak-monsoon
272,273,2025-12-13 19:00:00,2025-12-13 21:00:00,2025-12-13 20:00:00,2025-12-14 05:00:00,3,3,1,0.0,-2.718489,Type 2 weak-monsoon,Type 2 weak-monsoon
273,274,2025-12-16 23:00:00,2025-12-17 00:00:00,2025-12-17 00:00:00,2025-12-17 09:00:00,2,2,1,0.0,-2.597930,Type 2 weak-monsoon,Type 2 weak-monsoon


In [7]:
EXAMPLE_DATES = [
    "2018-02-03",
    "2023-01-24",
    ]

for target in EXAMPLE_DATES:
    print(f"\nTARGET {target} | raw catalog")
    display(
        catalog_df[
            catalog_df["event_start"].astype(str).str.startswith(target)
            | catalog_df["event_end"].astype(str).str.startswith(target)
            | catalog_df["event_peak"].astype(str).str.startswith(target)
        ][["event_start", "event_end", "event_peak", "duration_hours", "event_peak_D_1e5_s-1"]]
    )

    print(f"TARGET {target} | merged catalog")
    display(
        merged_catalog[
            merged_catalog["event_start"].astype(str).str.startswith(target)
            | merged_catalog["event_end"].astype(str).str.startswith(target)
            | merged_catalog["event_peak"].astype(str).str.startswith(target)
        ][[
            "event_start",
            "event_end",
            "event_peak",
            "duration_hours",
            "threshold_hit_hours",
            "merged_subevents_count",
            "max_internal_gap_hours",
            "event_peak_D_1e5_s-1",
        ]]
    )



TARGET 2018-02-03 | raw catalog


,event_start,event_end,event_peak,duration_hours,event_peak_D_1e5_s-1
201,2018-02-03 13:00:00,2018-02-04,2018-02-03 20:00:00,12,-3.08431


TARGET 2018-02-03 | merged catalog


,event_start,event_end,event_peak,duration_hours,threshold_hit_hours,merged_subevents_count,max_internal_gap_hours,event_peak_D_1e5_s-1
192,2018-02-03 13:00:00,2018-02-04,2018-02-03 20:00:00,12,12,1,0.0,-3.08431



TARGET 2023-01-24 | raw catalog


,event_start,event_end,event_peak,duration_hours,event_peak_D_1e5_s-1
252,2023-01-23 19:00:00,2023-01-24 08:00:00,2023-01-24 02:00:00,14,-4.078819


TARGET 2023-01-24 | merged catalog


,event_start,event_end,event_peak,duration_hours,threshold_hit_hours,merged_subevents_count,max_internal_gap_hours,event_peak_D_1e5_s-1
241,2023-01-23 19:00:00,2023-01-24 08:00:00,2023-01-24 02:00:00,14,14,1,0.0,-4.078819
